# Text embeddings

After chunking, RAG has to find the chunks that matter for a question. Keyword search matches words:
"bug" finds a software bug and an insect in a medical study alike. **Semantic search** matches meaning,
and it works on **embeddings**.

An embedding is a long list of numbers that stands for the meaning of a text. Texts with a similar
meaning get similar lists. Each number is a learned feature; what each one means is not something
you can read off.

Anthropic has no embedding model; the course uses **VoyageAI** (`voyageai` package, `VOYAGE_API_KEY`,
model `voyage-3-large`). Here the same Voyage models come through OpenRouter, so the key in `.env`
is enough: see [`embeddings.py`](./embeddings.py). The model is `voyage-4-large`, the newer version.

## Setup

The course function, extended the way the course itself does it in the next lesson: it also takes a list of texts.

In [1]:
import inspect
from pathlib import Path

from chunking import chunk_by_section
from embeddings import generate_embedding

print(inspect.getsource(generate_embedding))

def generate_embedding(chunks, model="voyageai/voyage-4-large", input_type="query"):
    """Embed one text (returns one vector) or a list of texts (returns a list of vectors).

    input_type: "query" for questions, "document" for the chunks being searched.
    """
    is_list = isinstance(chunks, list)
    response = httpx.post(
        EMBEDDINGS_URL,
        headers={"Authorization": f"Bearer {os.environ['ANTHROPIC_API_KEY']}"},
        json={"model": model, "input": chunks if is_list else [chunks], "input_type": input_type},
        timeout=60,
    )
    response.raise_for_status()
    embeddings = [item["embedding"] for item in sorted(response.json()["data"], key=lambda item: item["index"])]
    return embeddings if is_list else embeddings[0]



## Embed a chunk

The lesson's demo: chunk `report.md` by section and embed the first chunk.

In [2]:
chunks = chunk_by_section(Path("report.md").read_text())
embedding = generate_embedding(chunks[0])

print(chunks[0])
print(f"{len(embedding)} numbers: {[round(x, 4) for x in embedding[:5]]} ...")
print(f"min {min(embedding):.3f}, max {max(embedding):.3f}")

# **Annual Interdisciplinary Research Review: Cross-Domain Insights**
1024 numbers: [-0.0382, 0.0041, 0.0135, 0.0231, -0.0512] ...
min -0.124, max 0.112


## Similar meaning, similar numbers

A preview of the next lesson. **Cosine similarity** compares two embeddings: 1 means the same direction
(the same meaning), around 0 means unrelated. Voyage vectors have length 1, so it is just
the sum of the products.

The question from the lesson against the medical and the software sections. The chunks are embedded
with `input_type="document"`, the question with `"query"`: Voyage tunes the vectors for search that way.

In [3]:
def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b, strict=True))


question = "How many bugs did engineers fix this year?"
sections = [chunk for chunk in chunks if chunk.startswith(("## Section 1:", "## Section 2:", "## Section 10:"))]

question_embedding = generate_embedding(question, input_type="query")
section_embeddings = generate_embedding(sections, input_type="document")

print(question)
for section, section_embedding in zip(sections, section_embeddings, strict=True):
    print(f"  {cosine_similarity(question_embedding, section_embedding):.3f}  {section.splitlines()[0]}")

How many bugs did engineers fix this year?
  0.107  ## Section 1: Medical Research - Understanding XDR-471 Syndrome
  0.333  ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
  0.094  ## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011


None of the three sections uses the word "bug", yet the software section, about fixing recurring errors,
scores about three times higher than the other two: the match is on meaning, not on words.
The numbers are low on an absolute scale (a long section against a short question); what matters
is the ranking.

Next lesson: the full RAG flow, with a vector store and search over every chunk.